# Capstone: End-to-End Classical ML Project
## Customer Churn Prediction — Full Production Pipeline

This capstone project walks through a **complete, production-quality ML pipeline** from raw data to a deployed REST API.

You will build a **customer churn predictor** — a model that predicts whether a customer will cancel their subscription. This is one of the most common real-world ML problems.

### What You Will Learn
- Exploratory Data Analysis (EDA) with statistics and visualizations
- Data preprocessing: missing values, encoding, scaling
- Feature engineering: creating new features from existing ones
- Model selection: comparing multiple algorithms
- Hyperparameter tuning with cross-validation
- Handling class imbalance
- Model evaluation: beyond accuracy (AUC, precision-recall, calibration)
- Model explainability: feature importance and SHAP
- Saving and loading models for production
- Wrapping in a FastAPI service

### Pipeline Overview
```
Raw Data → EDA → Preprocessing → Feature Engineering
        → Model Selection → Hyperparameter Tuning
        → Final Evaluation → Save Model → Deploy API
```

## Real-World Analogy

Think of this end-to-end ML project like **building a car from scratch**:
- **EDA** = inspecting the raw materials before you start
- **Feature engineering** = machining parts to precise tolerances
- **Preprocessing pipeline** = the assembly line that transforms parts consistently
- **Cross-validation** = test-driving on multiple roads, not just one
- **Threshold optimisation** = calibrating the brakes for your specific use case
- **FastAPI serving** = opening the showroom so customers can drive it

Every step is necessary — skip one and the car breaks down in production.


## Step 1: Setup and Data Generation

In [ ]:
import numpy as np
import pandas as pd
import pickle
import warnings
import os
import time
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    roc_auc_score, classification_report, confusion_matrix,
    precision_recall_curve, average_precision_score, brier_score_loss
)
from sklearn.calibration import CalibratedClassifierCV

np.random.seed(42)

# ── Synthetic customer dataset ────────────────────────────────────────────────
def generate_customer_data(n=5000, seed=42):
    rng = np.random.RandomState(seed)

    age           = rng.randint(18, 75, n)
    tenure_months = rng.randint(1, 84, n)      # months as customer
    monthly_charge= rng.normal(65, 20, n).clip(20, 120).round(2)
    num_products  = rng.randint(1, 6, n)
    support_calls = rng.poisson(2, n)           # calls to customer support
    login_freq    = rng.poisson(8, n)           # logins per month
    contract_type = rng.choice(['Month-to-Month', 'One-Year', 'Two-Year'], n,
                               p=[0.55, 0.25, 0.20])
    payment_method= rng.choice(['Credit Card', 'Bank Transfer', 'Electronic Check'], n)
    internet_type = rng.choice(['Fiber', 'DSL', 'No'], n, p=[0.44, 0.34, 0.22])

    # Introduce 5% missing values in some columns
    for arr in [login_freq, support_calls]:
        arr[rng.choice(n, int(0.05 * n), replace=False)] = np.nan

    # Churn: higher with month-to-month, high support calls, fiber (expensive)
    logit = (
        -2.0
        + 0.8  * (contract_type == 'Month-to-Month').astype(float)
        - 0.5  * (contract_type == 'Two-Year').astype(float)
        + 0.15 * np.nan_to_num(support_calls)
        - 0.03 * tenure_months
        + 0.3  * (internet_type == 'Fiber').astype(float)
        - 0.02 * np.nan_to_num(login_freq)
        + rng.normal(0, 0.5, n)
    )
    churn = (1 / (1 + np.exp(-logit)) > 0.5).astype(int)

    df = pd.DataFrame({
        'customer_id':    [f'C{i:05d}' for i in range(n)],
        'age':            age,
        'tenure_months':  tenure_months,
        'monthly_charge': monthly_charge,
        'num_products':   num_products,
        'support_calls':  support_calls,
        'login_freq':     login_freq,
        'contract_type':  contract_type,
        'payment_method': payment_method,
        'internet_type':  internet_type,
        'churn':          churn,
    })
    return df

df = generate_customer_data(5000)
print(f"Dataset shape: {df.shape}")
print(f"Churn rate:    {df['churn'].mean():.1%}")
print()
print(df.head())
print()
print("Missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

## Step 2: Exploratory Data Analysis (EDA)

In [ ]:
print("=" * 60)
print("EXPLORATORY DATA ANALYSIS")
print("=" * 60)
print()

# Basic statistics
print("Numerical feature statistics:")
print(df.describe().round(2))
print()

# Churn rate by categorical features
print("Churn rate by contract type:")
print(df.groupby('contract_type')['churn'].agg(['mean', 'count'])
        .rename(columns={'mean': 'churn_rate', 'count': 'n_customers'})
        .sort_values('churn_rate', ascending=False))
print()

print("Churn rate by internet type:")
print(df.groupby('internet_type')['churn'].agg(['mean', 'count'])
        .rename(columns={'mean': 'churn_rate', 'count': 'n_customers'})
        .sort_values('churn_rate', ascending=False))
print()

# Numerical correlations with churn
num_cols = ['age', 'tenure_months', 'monthly_charge', 'num_products',
            'support_calls', 'login_freq']
corr = df[num_cols + ['churn']].corr()['churn'].drop('churn').sort_values()
print("Correlation with churn (most to least):")
for feat, corr_val in corr.items():
    bar = '#' * int(abs(corr_val) * 30)
    direction = '+' if corr_val > 0 else '-'
    print(f"  {feat:20s}: {direction}{bar} ({corr_val:+.3f})")

print()
print("Key insights from EDA:")
insights = [
    "Month-to-Month contracts have ~3× higher churn than Two-Year",
    "tenure_months negatively correlated: newer customers churn more",
    "support_calls positively correlated: frustrated customers leave",
    "Fiber internet type has highest churn (expensive, competitive)",
    "5% missing values in login_freq and support_calls — need imputation",
]
for ins in insights:
    print(f"  • {ins}")

## Step 3: Feature Engineering

In [ ]:
def engineer_features(df):
    """Create new features from existing ones."""
    df = df.copy()

    # Total lifetime value
    df['total_revenue'] = df['tenure_months'] * df['monthly_charge']

    # Revenue per product
    df['charge_per_product'] = df['monthly_charge'] / df['num_products']

    # Support call rate per tenure year
    df['support_call_rate'] = df['support_calls'] / (df['tenure_months'] / 12 + 0.1)

    # Engagement score (log-scaled login frequency)
    df['engagement'] = np.log1p(df['login_freq'].fillna(0))

    # High risk flag: month-to-month + high support calls
    df['high_risk'] = (
        (df['contract_type'] == 'Month-to-Month') &
        (df['support_calls'].fillna(0) > 3)
    ).astype(int)

    # Tenure group
    df['tenure_group'] = pd.cut(
        df['tenure_months'],
        bins=[0, 12, 24, 48, 84],
        labels=['0-1yr', '1-2yr', '2-4yr', '4+yr']
    ).astype(str)

    return df

df = engineer_features(df)
print("Features after engineering:")
new_features = ['total_revenue', 'charge_per_product', 'support_call_rate',
                'engagement', 'high_risk', 'tenure_group']
print(df[new_features].describe().round(2))
print()
print(f"High-risk customers churn at: {df[df['high_risk']==1]['churn'].mean():.1%}")
print(f"Normal customers churn at:    {df[df['high_risk']==0]['churn'].mean():.1%}")

## Step 4: Preprocessing Pipeline

In [ ]:
# Drop ID column — not a feature
feature_df = df.drop(columns=['customer_id'])

X = feature_df.drop('churn', axis=1)
y = feature_df['churn']

# Identify column types
numerical_cols   = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

print(f"Numerical features ({len(numerical_cols)}):   {numerical_cols}")
print(f"Categorical features ({len(categorical_cols)}): {categorical_cols}")
print()

# Preprocessing pipeline
num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),   # fill NaN with median
    ('scaler',  StandardScaler()),                    # normalize to zero mean
])

cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),  # fill NaN with mode
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('num', num_transformer, numerical_cols),
    ('cat', cat_transformer, categorical_cols),
])

# Train / validation / test split (70 / 15 / 15)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val,  y_train, y_val  = train_test_split(X_temp, y_temp, test_size=0.176, random_state=42, stratify=y_temp)

print(f"Train:      {len(X_train):,} rows (churn: {y_train.mean():.1%})")
print(f"Validation: {len(X_val):,} rows  (churn: {y_val.mean():.1%})")
print(f"Test:       {len(X_test):,} rows  (churn: {y_test.mean():.1%})")

## Step 5: Model Selection — Compare Multiple Algorithms

In [ ]:
print("=" * 60)
print("MODEL SELECTION")
print("=" * 60)
print()

# Class imbalance handling: use class_weight='balanced'
models = {
    'Logistic Regression': LogisticRegression(
        class_weight='balanced', max_iter=1000, random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100, random_state=42
    ),
}

# 5-fold stratified CV for fair comparison
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = {}
for name, model in models.items():
    pipe = Pipeline([('pre', preprocessor), ('clf', model)])
    t0 = time.time()
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    elapsed = time.time() - t0
    results[name] = {'mean': scores.mean(), 'std': scores.std(), 'time': elapsed}
    print(f"  {name:25s}: AUC = {scores.mean():.4f} ± {scores.std():.4f}  ({elapsed:.1f}s)")

best_model_name = max(results, key=lambda k: results[k]['mean'])
print()
print(f"Winner: {best_model_name} (AUC = {results[best_model_name]['mean']:.4f})")

## Step 6: Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

print("Hyperparameter tuning (RandomizedSearch + 5-fold CV)...")

gb_pipe = Pipeline([
    ('pre', preprocessor),
    ('clf', GradientBoostingClassifier(random_state=42)),
])

param_dist = {
    'clf__n_estimators':   randint(50, 400),
    'clf__max_depth':      randint(2, 8),
    'clf__learning_rate':  uniform(0.01, 0.29),
    'clf__subsample':      uniform(0.6, 0.4),
    'clf__min_samples_leaf': randint(1, 20),
}

t0 = time.time()
search = RandomizedSearchCV(
    gb_pipe,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    random_state=42,
    verbose=0,
)
search.fit(X_train, y_train)

print(f"Search complete in {time.time()-t0:.1f}s")
print(f"Best CV AUC:  {search.best_score_:.4f}")
print(f"Best params:")
for k, v in search.best_params_.items():
    print(f"  {k}: {v}")

# Best pipeline
best_pipeline = search.best_estimator_

# Validation set evaluation
val_probs = best_pipeline.predict_proba(X_val)[:, 1]
val_auc   = roc_auc_score(y_val, val_probs)
print(f"\nValidation AUC: {val_auc:.4f}")

## Step 7: Final Evaluation on Test Set

In [ ]:
print("=" * 60)
print("FINAL EVALUATION ON HELD-OUT TEST SET")
print("=" * 60)
print()

# Predictions
test_probs = best_pipeline.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= 0.5).astype(int)

# Core metrics
test_auc = roc_auc_score(y_test, test_probs)
test_ap  = average_precision_score(y_test, test_probs)
brier    = brier_score_loss(y_test, test_probs)

print(f"ROC-AUC:              {test_auc:.4f}")
print(f"Average Precision:    {test_ap:.4f}")
print(f"Brier Score:          {brier:.4f}  (0=perfect, 0.25=random)")
print()

print("Classification Report (threshold=0.5):")
print(classification_report(y_test, test_preds, target_names=['Stay', 'Churn']))

print("Confusion Matrix:")
cm = confusion_matrix(y_test, test_preds)
print(f"  TN={cm[0,0]:4d}  FP={cm[0,1]:4d}")
print(f"  FN={cm[1,0]:4d}  TP={cm[1,1]:4d}")
print()

# Business impact: choose threshold based on business cost
# Missing a churner (FN) costs $500 (lost customer)
# False alarm (FP) costs $20 (unnecessary retention offer)
FN_COST = 500
FP_COST = 20

thresholds = np.arange(0.1, 0.9, 0.05)
costs = []
for thresh in thresholds:
    preds_t = (test_probs >= thresh).astype(int)
    cm_t = confusion_matrix(y_test, preds_t)
    cost = cm_t[1, 0] * FN_COST + cm_t[0, 1] * FP_COST
    costs.append(cost)

best_thresh = thresholds[np.argmin(costs)]
min_cost    = min(costs)
print(f"Business-optimal threshold: {best_thresh:.2f}  (min cost: ${min_cost:,})")
preds_optimal = (test_probs >= best_thresh).astype(int)
print(classification_report(y_test, preds_optimal, target_names=['Stay', 'Churn']))

## Step 8: Feature Importance and Explainability

In [ ]:
print("Feature Importance (from GBM):")
print()

# Get feature names after preprocessing
fitted_pre  = best_pipeline.named_steps['pre']
num_names   = numerical_cols
cat_names   = fitted_pre.named_transformers_['cat'].named_steps['encoder'].get_feature_names_out(categorical_cols).tolist()
all_names   = num_names + cat_names

# GBM feature importances
clf = best_pipeline.named_steps['clf']
importances = clf.feature_importances_

feat_imp = pd.Series(importances, index=all_names).sort_values(ascending=False)

print("Top 15 features:")
for feat, imp in feat_imp.head(15).items():
    bar = '█' * int(imp * 200)
    print(f"  {feat:40s}: {bar} ({imp:.4f})")

print()

# SHAP (if available)
try:
    import shap
    X_test_transformed = fitted_pre.transform(X_test)
    explainer = shap.TreeExplainer(clf)
    shap_values = explainer.shap_values(X_test_transformed[:100])  # first 100 samples
    print("SHAP values computed (first 100 test samples)")
    print(f"SHAP array shape: {shap_values.shape}")
    print("(Use shap.summary_plot(shap_values, ...) for visualization)")
except ImportError:
    print("SHAP not installed. Install with: pip install shap")
    print("SHAP provides per-prediction explanations: why did model predict churn for this customer?")

## Step 9: Save Model for Production

In [ ]:
import tempfile

model_dir = tempfile.mkdtemp()
model_path = os.path.join(model_dir, 'churn_model.pkl')

# Save the full pipeline (preprocessing + model)
with open(model_path, 'wb') as f:
    pickle.dump(best_pipeline, f)

file_size_kb = os.path.getsize(model_path) / 1024
print(f"Model saved: {model_path}")
print(f"File size:   {file_size_kb:.1f} KB")
print()

# Load and verify
with open(model_path, 'rb') as f:
    loaded_pipeline = pickle.load(f)

# Verify predictions match
loaded_probs = loaded_pipeline.predict_proba(X_test[:5])[:, 1]
original_probs = best_pipeline.predict_proba(X_test[:5])[:, 1]
match = np.allclose(loaded_probs, original_probs)
print(f"Loaded model predictions match original: {match}")
print(f"Sample predictions: {loaded_probs.round(3)}")
print()
print("Production deployment tip:")
print("  Use mlflow.sklearn.save_model() or bentoml.sklearn.save_model()")
print("  for versioned, metadata-tracked model storage.")

# Also save model metadata
metadata = {
    'model_type': type(clf).__name__,
    'best_params': search.best_params_,
    'test_auc': float(test_auc),
    'test_ap':  float(test_ap),
    'optimal_threshold': float(best_thresh),
    'feature_count': len(all_names),
    'train_samples': len(X_train),
}
import json
meta_path = os.path.join(model_dir, 'metadata.json')
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print()
print("Model metadata:")
print(json.dumps(metadata, indent=2))

## Step 10: Production API (FastAPI)

Save this as `app.py` and run with `uvicorn app:app --reload`

In [ ]:
FASTAPI_CODE = '''
# app.py — save and run: uvicorn app:app --reload
import pickle
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel, Field
from typing import Literal

# Load model once at startup
with open('churn_model.pkl', 'rb') as f:
    MODEL = pickle.load(f)

THRESHOLD = 0.35  # business-optimal threshold

app = FastAPI(title="Churn Prediction API", version="1.0")

class CustomerFeatures(BaseModel):
    age: int = Field(..., ge=18, le=100)
    tenure_months: int = Field(..., ge=0, le=240)
    monthly_charge: float = Field(..., ge=0)
    num_products: int = Field(..., ge=1, le=10)
    support_calls: float = Field(default=0.0, ge=0)
    login_freq: float = Field(default=8.0, ge=0)
    contract_type: Literal['Month-to-Month', 'One-Year', 'Two-Year']
    payment_method: Literal['Credit Card', 'Bank Transfer', 'Electronic Check']
    internet_type: Literal['Fiber', 'DSL', 'No']

class PredictionResponse(BaseModel):
    churn_probability: float
    will_churn: bool
    risk_level: str
    recommendation: str

@app.post("/predict", response_model=PredictionResponse)
def predict(customer: CustomerFeatures):
    # Feature engineering (must match training)
    data = customer.dict()
    data['total_revenue']      = data['tenure_months'] * data['monthly_charge']
    data['charge_per_product'] = data['monthly_charge'] / data['num_products']
    data['support_call_rate']  = data['support_calls'] / (data['tenure_months'] / 12 + 0.1)
    import numpy as np
    data['engagement'] = np.log1p(data['login_freq'])
    data['high_risk'] = int(data['contract_type'] == 'Month-to-Month' and data['support_calls'] > 3)
    bins = [0, 12, 24, 48, 84]
    labels = ['0-1yr', '1-2yr', '2-4yr', '4+yr']
    for i, (lo, hi) in enumerate(zip(bins, bins[1:])):
        if lo <= data['tenure_months'] < hi:
            data['tenure_group'] = labels[i]
            break
    else:
        data['tenure_group'] = '4+yr'

    df = pd.DataFrame([data])
    prob = MODEL.predict_proba(df)[0, 1]

    if prob < 0.2:   risk, rec = 'Low',    'No action needed'
    elif prob < 0.5: risk, rec = 'Medium', 'Send retention email'
    elif prob < 0.7: risk, rec = 'High',   'Offer discount'
    else:            risk, rec = 'Critical','Immediate outreach required'

    return PredictionResponse(
        churn_probability=round(float(prob), 4),
        will_churn=prob >= THRESHOLD,
        risk_level=risk,
        recommendation=rec,
    )

@app.get("/health")
def health():
    return {"status": "ok", "model": type(MODEL.named_steps["clf"]).__name__}
'''

print("FastAPI Service (app.py):")
print(FASTAPI_CODE)

print("Run with: uvicorn app:app --reload")
print()
print("Test with curl:")
print("  curl -X POST http://localhost:8000/predict \\")
print("    -H 'Content-Type: application/json' \\")
print("    -d '{\"age\": 45, \"tenure_months\": 3, \"monthly_charge\": 85,")
print("         \"num_products\": 1, \"support_calls\": 5, \"login_freq\": 2,")
print("         \"contract_type\": \"Month-to-Month\",")
print("         \"payment_method\": \"Electronic Check\",")
print("         \"internet_type\": \"Fiber\"}' ")

## Interview Questions & Answers

---

**Q1: What is data leakage in an ML pipeline and how does sklearn's Pipeline prevent it?**

A: Data leakage is when information from the validation/test set "leaks" into the training process, making evaluation metrics look better than real-world performance. Common examples: fitting a StandardScaler on the full dataset before splitting (the scaler learns test-set statistics); computing target-based features (mean-encoding) before splitting. sklearn's `Pipeline` prevents this by ensuring `.fit_transform()` is called only on training data, while `.transform()` (no fitting) is applied to validation/test data. The pipeline stores the fitted parameters and reuses them faithfully.

---

**Q2: Why use StratifiedKFold instead of regular KFold for classification?**

A: Regular KFold splits data randomly — with a 5% positive rate dataset, some folds might have 2% positives and others 8%, leading to high variance in evaluation metrics. `StratifiedKFold` guarantees each fold has approximately the same class ratio as the full dataset. This is especially important for imbalanced datasets where a fold with too few positives produces unreliable AUC/precision estimates. Always use stratified splitting for classification tasks.

---

**Q3: A stakeholder says "your model has 95% accuracy so it must be good." What do you tell them?**

A: Accuracy is misleading on imbalanced datasets. With 5% fraud rate, a model that predicts "not fraud" for every transaction has 95% accuracy — and catches zero frauds. Better metrics: **Precision** (of predicted positives, how many are real?), **Recall** (of real positives, how many did we catch?), **F1** (harmonic mean), **ROC-AUC** (ranking ability), **Average Precision** (area under precision-recall curve). More importantly: frame it in **business cost** — what does a missed fraud (FN) cost vs a false alarm (FP)? Threshold optimisation ties the model directly to business outcomes.

---

**Q4: How do you handle missing values in a production ML pipeline?**

A: The correct approach: (1) Understand WHY values are missing — Missing Not At Random (MNAR) patterns can be informative features themselves (e.g., "income not reported" may correlate with default). (2) Use `SimpleImputer` inside the sklearn Pipeline so the imputation statistics are learned on training data only. (3) For numeric: median imputation (robust to outliers) or model-based imputation. (4) For categorical: most-frequent or a dedicated "Unknown" category. (5) Add a missingness indicator binary feature for MNAR cases. Never drop columns just because they have nulls — they may be your most predictive features.

---

**Q5: Why save metadata alongside the model pickle?**

A: A pickle file alone tells you nothing about the model. Six months later: which features does it expect? What was its test AUC? What threshold should production use? The metadata JSON stores: feature names and types (so the API knows what to expect), evaluation metrics (for monitoring drift — alert if live AUC drops 5% from baseline), optimal decision threshold, training data size, random seed, and timestamp. This is the difference between a research prototype and a production-grade system.

---

**Q6: What is the difference between a model pipeline and MLOps?**

A: A **model pipeline** (this notebook) is the code that transforms data, trains the model, and produces predictions. It runs once (or periodically). **MLOps** is the engineering discipline of running model pipelines reliably in production: automated retraining when data drifts, CI/CD for model updates, A/B testing new versions, monitoring prediction distributions, rollback if a model degrades, feature stores for consistent feature computation, lineage tracking. Think of this notebook as one component inside a much larger MLOps system.

## Recommended Resources

| Resource | Link | Why |
|---|---|---|
| sklearn Pipeline Guide | https://scikit-learn.org/stable/modules/compose.html | Deep dive into pipelines |
| SHAP Library | https://shap.readthedocs.io/ | Explainability |
| Imbalanced-learn | https://imbalanced-learn.org/ | SMOTE and other samplers |
| Kaggle Churn Datasets | https://www.kaggle.com/datasets?search=churn | Practice data |
| FastAPI Docs | https://fastapi.tiangolo.com/ | Serving |
| MLflow | https://mlflow.org/ | Experiment tracking |


## Summary: Production ML Checklist

```
✅ EDA: understand data, find correlations, spot missing values
✅ Feature engineering: domain-informed new features
✅ Preprocessing pipeline: imputation + scaling + encoding in Pipeline
✅ Train/val/test split: stratified, 70/15/15
✅ Model selection: cross-validated comparison of multiple algorithms
✅ Hyperparameter tuning: RandomizedSearch > GridSearch for efficiency
✅ Class imbalance: class_weight='balanced' and/or business-optimal threshold
✅ Evaluation beyond accuracy: AUC, AP, Brier score, business cost
✅ Explainability: feature importance, SHAP for individual predictions
✅ Save full pipeline (preprocessing + model together in one pkl)
✅ Save metadata: params, metrics, threshold, feature count
✅ Production API: FastAPI with Pydantic validation + feature engineering
```

### Key Lessons
1. **Always use a Pipeline** — preprocessing steps must be fit on train only, applied to all sets consistently
2. **Stratify your splits** — especially important for imbalanced datasets
3. **Business metrics matter** — optimize threshold for business cost, not just AUC
4. **Explain your model** — feature importance builds trust with stakeholders
5. **Save everything** — model, metadata, and threshold together

### Next Capstone
- **End_to_End_DL_Project**: same rigor applied to a neural network (image classification)
- **LLM_Application_Project**: build a RAG chatbot with document retrieval